In [1]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

In [2]:
def add_numbers(a: float, b: float) -> float:
    """Adds two floating-point numbers together."""
    return a + b


def multiply_numbers(a: float, b: float) -> float:
    """Multiplies two floating-point numbers together."""
    return a * b

In [3]:
tools_map = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers,
}

In [4]:
def run_agent(user_prompt: str):
    print(f"User: {user_prompt}\n")

    # 3. Configure the Agent's tools and instructions
    config = types.GenerateContentConfig(
        system_instruction="You are a helpful calculation assistant. Always use your provided tools to perform math.",
        tools=[add_numbers, multiply_numbers],
        temperature=0.0,
        automatic_function_calling={"disable": True},
    )

    chat = client.chats.create(model="gemini-3-flash-preview", config=config)
    response = chat.send_message(user_prompt)

    while response.function_calls:
        for function_call in response.function_calls:
            name = function_call.name
            args = function_call.args

            print(
                f"🤖 [Agent Decision]: Needs to call function '{name}' with arguments {args}"
            )

            # Execute the local Python function dynamically
            if name in tools_map:
                tool_result = tools_map[name](**args)
                print(f"🔌 [Tool Output]: Result from {name} = {tool_result}\n")
            else:
                tool_result = f"Error: Tool {name} not found."

            # Feed the tool execution results back into the chat session
            response = chat.send_message(
                types.Part.from_function_response(
                    name=name, response={"result": tool_result}
                )
            )

    # Print the final compiled response once the agent completes its work
    print(f"🤖 Final Agent Answer: {response.text}")

In [5]:
run_agent("First add 15 and 35. Then, take that result and multiply it by 4.")

User: First add 15 and 35. Then, take that result and multiply it by 4.

🤖 [Agent Decision]: Needs to call function 'add_numbers' with arguments {'b': 35, 'a': 15}
🔌 [Tool Output]: Result from add_numbers = 50

🤖 [Agent Decision]: Needs to call function 'multiply_numbers' with arguments {'b': 4, 'a': 50}
🔌 [Tool Output]: Result from multiply_numbers = 200

🤖 Final Agent Answer: The result of adding 15 and 35 is 50. Multiplying that result by 4 gives 200.


In [6]:
run_agent("Add 2 and 4 and then add the result to 100")

User: Add 2 and 4 and then add the result to 100

🤖 [Agent Decision]: Needs to call function 'add_numbers' with arguments {'b': 4, 'a': 2}
🔌 [Tool Output]: Result from add_numbers = 6

🤖 [Agent Decision]: Needs to call function 'add_numbers' with arguments {'a': 6, 'b': 100}
🔌 [Tool Output]: Result from add_numbers = 106

🤖 Final Agent Answer: The sum of 2 and 4 is 6, and adding that to 100 gives a final result of 106.
